# Level 2 — COCO 2014, seeds 42 / 44 / 46, Beam Search k=3

Chạy lần lượt **6 cell code** trên GPU Colab; tại Cell 2 chọn `42`, hoàn tất rồi đổi `44`, rồi `46`. Giữ `EXPERIMENT_TAG="main"`. Mỗi seed chỉ chạy một phiên. Seed 42 chỉ **đánh giá lại checkpoint R1 lịch sử**, không train; seed 44/46 fine-tune R1 từ **cùng checkpoint Level 1 seed 42 lịch sử**.

Đây là biến thiên của giai đoạn fine-tune Level 2 **có điều kiện trên upstream cố định**; không phải ba pipeline end-to-end với seed được ghép xuyên các level. Config R1 lịch sử trên Drive xác nhận weighted-sum alpha=0.5, retrieval top50→5, **30 epoch tối đa**, AdamW **lr3e-4 / wd1e-4**, batch32, patience3. Nhiệt độ1, min_delta1e-4, dynamic padding và token-weighted CE lấy từ mã v6 đang có; config lịch sử không ghi loss aggregation/min_delta, nên chưa thể khẳng định khôi phục chính xác hoàn toàn lần train cũ.

Lần train seed42 lịch sử dùng NVIDIA L4 / BF16 / TF32. Notebook mới dùng BF16 và TF32 nếu GPU hỗ trợ; precision thực tế được khóa vào protocol. Thay GPU có thể khác precision và tạo run hash khác. Muốn ba seed cùng recipe/môi trường tuyệt đối cần nguồn training lịch sử đầy đủ hoặc train lại cả seed42 theo một protocol mới; notebook này chủ động giữ checkpoint42 của bạn.

Beam search dùng đúng quy tắc Level 1 mới: k3, alpha0.6, max30 **gồm BOS**, chặn PAD/BOS, xếp hạng cả beam kết thúc và đang mở rộng bằng normalized cumulative log-probability. METEOR dùng Java COCO; không so trực tiếp với METEOR-NLTK cũ.

Input checkpoint, vocab, annotations, CLIP feature, KB và retrieval cache cũ chỉ được **đọc**. Đầu ra riêng: `Data/baseline_level2/multiseed_beam3_v1/run_<hash>/seed_<seed>/`. File hoàn tất khác nội dung/cấu hình được giữ nguyên và dừng. Train lưu best/last đầy đủ RNG; ngắt giữa epoch chạy lại epoch đó. Decode lưu mỗi100 ảnh. Chỉ bật `OLD_SESSION_STOPPED` khi phiên cũ cùng seed đã dừng hẳn.

Cache retrieval cũ được xác minh metadata, shape, row IDs, self-exclusion và cosine score. Nếu thiếu cache, truy hồi FAISS **trong RAM**, có thể mất nhiều thời gian; không tạo cache hay index trên Drive nguồn. Chỉ có checkpoint, predictions, metrics và summary kết quả mới ở vùng output. Notebook chưa được chạy toàn bộ training/evaluation trên GPU Colab; cài thư viện và mount Drive cần Internet/quyền tài khoản của bạn.


In [1]:
#@title CELL 1 — Cài thư viện, Java 11 và kiểm tra bộ chấm điểm
"""Level 2: seed42 historical R1 evaluation; seeds44/46 conditional fine-tuning.
Inputs read-only; guarded per-seed outputs. Full instructions in the first markdown cell.
"""
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, subprocess, shutil, tempfile, tarfile, urllib.request, zipfile
from pathlib import Path
from importlib import metadata

RUNTIME_DIR = Path("/content/level2_beam3_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pycocoevalcap==1.2", "nltk>=3.9,<4", "tqdm>=4.66,<5", "faiss-cpu>=1.8,<2"],
    check=True,
)

def java11_version(java):
    result = subprocess.run([str(java), "-version"], capture_output=True, text=True)
    return result.returncode == 0 and 'version "11.' in result.stderr

java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java"))
java_candidates += list((RUNTIME_DIR / "java11").glob("*/bin/java"))
if not any(java11_version(p) for p in java_candidates):
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq",
                        "openjdk-11-jre-headless"], check=True)
    except subprocess.CalledProcessError:
        # Colab Ubuntu mới có thể không cung cấp package OpenJDK 11.
        java_root = RUNTIME_DIR / "java11"
        java_root.mkdir(exist_ok=True)
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            archive = Path(tmp) / "jdk11.tar.gz"
            urllib.request.urlretrieve(
                "https://api.adoptium.net/v3/binary/latest/11/ga/linux/x64/jre/hotspot/normal/eclipse",
                archive,
            )
            with tarfile.open(archive, "r:gz") as tar:
                tar.extractall(java_root, filter="data")
    java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java"))
    java_candidates += list((RUNTIME_DIR / "java11").glob("*/bin/java"))
JAVA_BIN = next((p for p in java_candidates if java11_version(p)), None)
if JAVA_BIN is None:
    raise RuntimeError("Không cài được Java 11; xem log cài đặt ở trên.")
os.environ["JAVA_HOME"] = str(JAVA_BIN.parent.parent)
os.environ["PATH"] = str(JAVA_BIN.parent) + os.pathsep + os.environ["PATH"]
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

import nltk
for resource in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f"Không tải được NLTK resource: {resource}")

import pycocoevalcap.spice.spice as spice_module
SPICE_DIR = Path(spice_module.__file__).parent
SPICE_LIB = SPICE_DIR / "lib"
SPICE_LIB.mkdir(exist_ok=True)
# Tải cả code jar và models jar qua HTTPS, tránh downloader HTTP cũ.
for suffix in ("", "-models"):
    jar_name = f"stanford-corenlp-3.6.0{suffix}.jar"
    jar_path = SPICE_LIB / jar_name
    if not jar_path.exists() or not zipfile.is_zipfile(jar_path):
        url = ("https://repo.maven.apache.org/maven2/edu/stanford/nlp/"
               f"stanford-corenlp/3.6.0/{jar_name}")
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            downloaded = Path(tmp) / jar_name
            urllib.request.urlretrieve(url, downloaded)
            if not zipfile.is_zipfile(downloaded):
                raise RuntimeError(f"Jar tải về không hợp lệ: {jar_name}")
            shutil.copyfile(downloaded, jar_path)

# Package và cache SPICE đều nằm trong runtime; không ghi công cụ lên Drive.
spice_module.TEMP_DIR = str(RUNTIME_DIR / "spice_tmp")
spice_module.CACHE_DIR = str(RUNTIME_DIR / "spice_cache")
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

class ManagedMeteor(Meteor):
    def close(self):
        process = getattr(self, "meteor_p", None)
        if process is not None and process.poll() is None:
            try:
                if process.stdin is not None:
                    process.stdin.close()
            except (OSError, ValueError):
                pass
            finally:
                process.kill()
                process.wait(timeout=30)
    def __del__(self):
        self.close()

def close_meteor(scorer):
    scorer.close()

_smoke_gts = PTBTokenizer().tokenize({1: [{"caption": "a cat sits on a mat"}]})
_smoke_res = PTBTokenizer().tokenize({1: [{"caption": "a cat sits on a mat"}]})
_meteor = ManagedMeteor()
try:
    _meteor_score, _ = _meteor.compute_score(_smoke_gts, _smoke_res)
finally:
    close_meteor(_meteor)
del _meteor
_spice_score, _ = Spice().compute_score(_smoke_gts, _smoke_res)
import math
if not all(math.isfinite(float(x)) for x in (_meteor_score, _spice_score)):
    raise RuntimeError("Smoke test METEOR/SPICE thất bại.")
print(f"Bộ chấm điểm OK • METEOR={_meteor_score:.4f} • SPICE={_spice_score:.4f}")
print("pycocoevalcap", metadata.version("pycocoevalcap"), "| NLTK", nltk.__version__)
# Nguồn triển khai: https://github.com/salaniz/pycocoevalcap


openjdk version "11.0.32.1" 2026-08-18
OpenJDK Runtime Environment (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)

Bộ chấm điểm OK • METEOR=1.0000 • SPICE=1.0000
pycocoevalcap 1.2 | NLTK 3.9.1


In [6]:
#@title CELL 2 — Chọn seed, Mount Drive và kiểm tra dữ liệu (chỉ đọc dữ liệu gốc)
# Chạy riêng 42, rồi đổi sang 44, rồi 46; mỗi lần chạy lại lần lượt 6 cell.
SEED_TO_RUN = "44" #@param ["42", "44", "46"]
# Chỉ bật nếu phiên Colab cũ cùng seed đã dừng hẳn nhưng còn khóa sau disconnect.
OLD_SESSION_STOPPED = True #@param {type:"boolean"}
SELECTED_SEED = int(SEED_TO_RUN)
# Giữ cùng tag cho 3 seed. Chỉ đổi tag khi muốn một thí nghiệm MỚI độc lập.
EXPERIMENT_TAG = "main" #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import json, re, random, hashlib, platform, time, gc
from collections import defaultdict
if not re.fullmatch(r"[A-Za-z0-9_-]{1,40}", EXPERIMENT_TAG):
    raise ValueError("EXPERIMENT_TAG chỉ gồm chữ không dấu, số, gạch dưới/gạch ngang, tối đa 40 ký tự.")
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import faiss
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display, Markdown

DATA_ROOT = Path("/content/drive/MyDrive/DataKhoaLuan/Data")
BASE_DIR = DATA_ROOT / "baseline_level1"
FEATURE_DIR = BASE_DIR / "clip_image_features"
LEVEL_DIR = DATA_ROOT / "baseline_level2"
OUTPUT_ROOT = LEVEL_DIR / "multiseed_beam3_v1"
UPSTREAM_CHECKPOINT = BASE_DIR / "checkpoints/checkpoint_best.pt"
EXPECTED_UPSTREAM_SHA256 = "9e9a453975270bb3d5480615cba9a4fed843ff1378b95c7a63b3db4820578dfe"
EXPECTED_SEED42_SHA256 = "5be25c5eaad79ed994c374c28834499be5441fd7ad1dad56912985c9a7fbae54"
# Verified source folder on Drive. Do not silently select another checkpoint.
SEED42_CHECKPOINT = LEVEL_DIR / "rag_outputs/R1_K5_N50_A0.5_seed42_L4/checkpoint_best.pt"
SOURCE_CACHE_DIR = LEVEL_DIR / "rag_level2_v6/r1_weighted_sum_k5_n50_a0p5_seed42/retrieval_cache"
TOP_N, TOP_K, FUSION_ALPHA, RETRIEVAL_TEMPERATURE = 50, 5, 0.5, 1.0
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_TF32 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
USE_FUSED_ADAMW = torch.cuda.is_available()
torch.backends.cuda.matmul.allow_tf32 = USE_TF32
torch.backends.cudnn.allow_tf32 = USE_TF32
MAX_LEN, BATCH_SIZE = 30, 32
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Hãy bật GPU trong Colab trước khi train/đánh giá toàn bộ dữ liệu.")

def read_json(path):
    with Path(path).open(encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True,
        ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()

vocab_path = BASE_DIR / "vocab.json"
vocab = read_json(vocab_path)
stoi = {str(k): int(v) for k, v in vocab["stoi"].items()}
raw_itos = vocab["itos"]
itos = ([str(x) for x in raw_itos] if isinstance(raw_itos, list)
        else [str(raw_itos[str(i)]) for i in range(len(stoi))])
VOCAB_SIZE = len(stoi)
assert VOCAB_SIZE == 16214, f"Vocab hiện tại khác dữ liệu đã kiểm tra: {VOCAB_SIZE}"
assert sorted(stoi.values()) == list(range(VOCAB_SIZE))
assert all(stoi.get(t) == i for i, t in enumerate(("<PAD>", "<BOS>", "<EOS>", "<UNK>")))
assert all(itos[i] == t for t, i in stoi.items()), "stoi/itos không khớp."
assert int(vocab.get("max_length", MAX_LEN)) == MAX_LEN

# Đường dẫn thực tế đã kiểm tra trên Drive, kèm đường dẫn ngắn trong yêu cầu.
annotation_candidates = [
    DATA_ROOT / "COCO_2014/annotations_trainval2014/annotations",
    DATA_ROOT / "COCO_2014/annotations",
]
ANNOTATION_DIR = next((p for p in annotation_candidates if all(
    (p / f"captions_karpathy_{s}.json").is_file()
    for s in ("train", "val", "test"))), None)
if ANNOTATION_DIR is None:
    raise FileNotFoundError("Không tìm được đủ 3 annotations Karpathy tại: "
                            + ", ".join(map(str, annotation_candidates)))
if not SEED42_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Thiếu checkpoint seed 42: {SEED42_CHECKPOINT}")

TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
def encode_caption(text):
    words = TOKEN_PATTERN.findall(text.lower())[:MAX_LEN - 2]
    values = [BOS_ID] + [stoi.get(w, UNK_ID) for w in words] + [EOS_ID]
    return torch.tensor(values, dtype=torch.long)

class CaptionDataset(Dataset):
    def __init__(self, features, image_ids, annotations):
        self.features = features
        row_by_id = {image_id: i for i, image_id in enumerate(image_ids)}
        grouped = defaultdict(list)
        for annotation in annotations:
            if str(annotation["caption"]).strip():
                grouped[int(annotation["image_id"])].append(annotation)
        # Giữ thứ tự image/caption của dataset baseline cũ trước khi shuffle.
        usable = [a for image_id in sorted(image_ids) for a in grouped[image_id]]
        self.feature_rows = np.asarray(
            [row_by_id[int(a["image_id"])] for a in usable], dtype=np.int32)
        self.tokens = [encode_caption(a["caption"]) for a in usable]
    def __len__(self):
        return len(self.feature_rows)
    def __getitem__(self, index):
        feature = torch.from_numpy(self.features[self.feature_rows[index]])
        caption = self.tokens[index]
        return feature, caption, int(self.feature_rows[index])

def collate_caption(batch):
    visual, sequences, rows = zip(*batch)
    return (torch.stack(visual),
            pad_sequence(sequences, batch_first=True, padding_value=PAD_ID),
            torch.tensor(rows, dtype=torch.long))

class ImageDataset(Dataset):
    def __init__(self, features, image_ids):
        self.features, self.image_ids = features, image_ids
    def __len__(self):
        return len(self.image_ids)
    def __getitem__(self, index):
        return self.image_ids[index], torch.from_numpy(self.features[index])

features, image_ids, caption_datasets = {}, {}, {}
references = None
DATA_FINGERPRINTS = {"vocab": sha256_file(vocab_path)}
expected_images = {"train": 113287, "val": 5000, "test": 5000}
expected_captions = {"train": 566747, "val": 25010, "test": 25010}
for split in ("train", "val", "test"):
    feature_path = FEATURE_DIR / f"{split}_features.npy"
    id_path = FEATURE_DIR / f"{split}_image_ids.json"
    annotation_path = ANNOTATION_DIR / f"captions_karpathy_{split}.json"
    for p in (feature_path, id_path, annotation_path):
        if not p.is_file():
            raise FileNotFoundError(p)
        DATA_FINGERPRINTS[f"{split}/{p.name}"] = sha256_file(p)
    # Nạp một lần vào RAM; không đọc ngẫu nhiên trên Drive trong từng minibatch.
    array = np.load(feature_path, allow_pickle=False)
    ids = [int(x) for x in read_json(id_path)]
    payload = read_json(annotation_path)
    annotations = payload["annotations"]
    assert array.shape == (expected_images[split], 512), (split, array.shape)
    assert array.dtype == np.float32 and np.isfinite(array).all(), split
    assert len(ids) == len(set(ids)) == len(array), f"IDs trùng/lệch: {split}"
    assert set(ids) == {int(a["id"]) for a in payload["images"]}, split
    assert ids == sorted(ids), f"Original L2 dataset requires sorted cached IDs: {split}"
    refs = defaultdict(list)
    for a in annotations:
        if str(a["caption"]).strip():
            refs[int(a["image_id"])].append(str(a["caption"]))
    assert set(ids) == set(refs), f"Thiếu/thừa caption tham chiếu: {split}"
    assert sum(map(len, refs.values())) == expected_captions[split], split
    assert np.allclose(np.linalg.norm(array, axis=1), 1.0, atol=1e-3), (
        f"Cache {split} không còn là đặc trưng CLIP đã chuẩn hóa.")
    features[split], image_ids[split] = array, ids
    if split in ("train", "val"):
        caption_datasets[split] = CaptionDataset(array, ids, annotations)
    else:
        references = dict(refs)
    print(f"{split}: {len(ids):,} ảnh • {len(annotations):,} captions • {array.shape}")
    del payload, annotations, refs

assert not (set(image_ids["train"]) & set(image_ids["val"]))
assert not (set(image_ids["train"]) & set(image_ids["test"]))
assert not (set(image_ids["val"]) & set(image_ids["test"]))

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

def make_train_loader(seed):
    generator = torch.Generator().manual_seed(seed)
    # num_workers=0 giúp lưu/khôi phục RNG chính xác giữa các epoch.
    loader = DataLoader(caption_datasets["train"], batch_size=BATCH_SIZE,
        shuffle=True, drop_last=False, num_workers=0, pin_memory=True,
        generator=generator, collate_fn=collate_caption)
    return loader, generator

train_loader, train_generator = make_train_loader(44)
val_loader = DataLoader(caption_datasets["val"], batch_size=BATCH_SIZE,
    shuffle=False, drop_last=False, num_workers=0, pin_memory=True,
    collate_fn=collate_caption)
test_loader = DataLoader(ImageDataset(features["test"], image_ids["test"]),
    batch_size=1, shuffle=False, drop_last=False, num_workers=0, pin_memory=True)
print(f"Vocab={VOCAB_SIZE:,} • GPU={torch.cuda.get_device_name()} • annotations={ANNOTATION_DIR}")
print("Seed được chọn:", SELECTED_SEED)
print("Vùng đầu ra mới (sẽ xác định sau khi kiểm tra protocol):", OUTPUT_ROOT)


# Pin historical upstream and historical R1 checkpoint before any output writes.
for path, expected_hash in ((UPSTREAM_CHECKPOINT, EXPECTED_UPSTREAM_SHA256),
                            (SEED42_CHECKPOINT, EXPECTED_SEED42_SHA256)):
    if not path.is_file():
        raise FileNotFoundError(f"Missing verified source checkpoint: {path}")
    if sha256_file(path) != expected_hash:
        raise RuntimeError(f"Source checkpoint changed; stop without output writes: {path}")
UPSTREAM_PAYLOAD = torch.load(UPSTREAM_CHECKPOINT, map_location="cpu", weights_only=True)
if UPSTREAM_PAYLOAD.get("config", {}).get("seed") != 42 or UPSTREAM_PAYLOAD.get("stoi") != stoi:
    raise ValueError("Historical Level1 checkpoint does not confirm seed42/vocab.")
upstream_itos = UPSTREAM_PAYLOAD.get("itos")
if isinstance(upstream_itos, dict):
    upstream_itos = [upstream_itos[str(i)] for i in range(VOCAB_SIZE)]
if upstream_itos != itos:
    raise ValueError("Historical upstream itos mismatch.")
SOURCE_CONFIG_PATH = BASE_DIR / "config.json"
DATA_FINGERPRINTS["baseline_config"] = sha256_file(SOURCE_CONFIG_PATH)
DATA_FINGERPRINTS["upstream_checkpoint"] = EXPECTED_UPSTREAM_SHA256
DATA_FINGERPRINTS["historical_R1_checkpoint"] = EXPECTED_SEED42_SHA256
HISTORICAL_CONFIG_PATH = SEED42_CHECKPOINT.parent / "config_R1.json"
historical_config = read_json(HISTORICAL_CONFIG_PATH)
for key,value in {"seed":42,"top_n":50,"top_k":5,"alpha":0.5,
        "caption_batch_size":32,"epochs":30,"learning_rate":3e-4,
        "weight_decay":1e-4,"early_stopping_patience":3,
        "b0_checkpoint_hash":EXPECTED_UPSTREAM_SHA256}.items():
    if historical_config.get(key) != value:
        raise ValueError(f"Historical L2 config mismatch: {key}")
DATA_FINGERPRINTS["historical_R1_config"] = sha256_file(HISTORICAL_CONFIG_PATH)

# The original KB contains exactly train/restval captions and no val/test images.
KB_CANDIDATES = [DATA_ROOT / "knowledge_base_level1_karpathy",
    Path("/content/drive/MyDrive/Data/Data/knowledge_base_level1_karpathy"),
    Path("/content/drive/MyDrive/Data/knowledge_base_level1_karpathy")]
KB_DIR = next((p for p in KB_CANDIDATES if all((p/n).is_file() for n in
    ("text_embeddings_train_restval.npy", "text_embedding_mapping.jsonl"))), None)
if KB_DIR is None:
    raise FileNotFoundError("Knowledge base not found at the original candidate paths.")
full_train_ids, full_val_ids, full_test_ids = (image_ids[s] for s in ("train", "val", "test"))
def file_sha256(path):
    return sha256_file(path) if Path(path).is_file() else None
def hash_id_list(ids):
    return hashlib.sha256(json.dumps([int(x) for x in ids], separators=(",", ":")).encode()).hexdigest()
load_json = read_json
MATERIALIZE_KB_EMBEDDINGS_IN_RAM = SELECTED_SEED in (44,46)
faiss.omp_set_num_threads(max(1, min(8, os.cpu_count() or 2)))
KB_EMBEDDING_PATH = KB_DIR / "text_embeddings_train_restval.npy"
KB_MAPPING_PATH = KB_DIR / "text_embedding_mapping.jsonl"
KB_METADATA_PATH = KB_DIR / "metadata.json"
KB_INDEX_CANDIDATES = [
    KB_DIR / "text_embeddings_train_restval.faiss",
    KB_DIR / "faiss_index_train_restval.index",
]

kb_embeddings = np.load(KB_EMBEDDING_PATH, mmap_mode="r")
assert kb_embeddings.shape == (566747, 512), kb_embeddings.shape
assert kb_embeddings.dtype == np.float32

kb_image_ids = np.empty(kb_embeddings.shape[0], dtype=np.int64)
kb_splits = []
kb_captions = []
with KB_MAPPING_PATH.open("r", encoding="utf-8") as handle:
    for position, line in enumerate(tqdm(handle, total=566747, desc="Load KB mapping")):
        row = json.loads(line)
        if position >= len(kb_image_ids):
            raise AssertionError("KB mapping có nhiều dòng hơn embedding")
        row_id = row.get("row_id", position)
        assert int(row_id) == position, f"Mapping row_id lệch tại {position}: {row_id}"
        split = str(row.get("split", ""))
        assert split in {"train", "restval"}, f"KB chứa split cấm: {split}"
        caption = str(row.get("caption", "")).strip()
        assert caption, f"KB caption rỗng tại row {position}"
        kb_image_ids[position] = int(row["image_id"])
        kb_splits.append(split)
        kb_captions.append(caption)

assert len(kb_captions) == kb_embeddings.shape[0]
assert set(kb_splits) <= {"train", "restval"}
assert len(set(kb_image_ids.tolist())) == 113287
assert set(kb_image_ids.tolist()) == set(full_train_ids)
assert set(kb_image_ids.tolist()).isdisjoint(full_val_ids)
assert set(kb_image_ids.tolist()).isdisjoint(full_test_ids)

probe_rows = np.linspace(0, len(kb_embeddings) - 1, num=10000, dtype=np.int64)
probe_vectors = np.asarray(kb_embeddings[probe_rows], dtype=np.float32)
assert np.isfinite(probe_vectors).all()
probe_norms = np.linalg.norm(probe_vectors, axis=1)
assert np.allclose(probe_norms, 1.0, atol=1e-3), (probe_norms.min(), probe_norms.max())

kb_index_path = next((path for path in KB_INDEX_CANDIDATES if path.is_file()), None)
if kb_index_path:
    kb_index = faiss.read_index(str(kb_index_path))
    if not isinstance(kb_index, faiss.IndexFlatIP):
        raise ValueError("L2 source retrieval expects exact IndexFlatIP.")
else:
    print("Không có FAISS index cache; xây IndexFlatIP trong RAM...")
    kb_index = faiss.IndexFlatIP(512)
    for start in tqdm(range(0, len(kb_embeddings), 8192), desc="Build FAISS RAM index"):
        batch = np.asarray(kb_embeddings[start:start + 8192], dtype=np.float32)
        kb_index.add(batch)

assert kb_index.d == 512
assert kb_index.ntotal == len(kb_embeddings)

index_probes = np.linspace(0, kb_index.ntotal - 1, num=5, dtype=np.int64)
reconstructed = np.vstack([kb_index.reconstruct(int(i)) for i in index_probes])
expected = np.asarray(kb_embeddings[index_probes], dtype=np.float32)
assert np.allclose(reconstructed, expected, atol=1e-5), "FAISS index không khớp embedding file"

KB_FILE_HASHES = {
    "embeddings_sha256": file_sha256(KB_EMBEDDING_PATH),
    "mapping_sha256": file_sha256(KB_MAPPING_PATH),
    "index_sha256": file_sha256(kb_index_path) if kb_index_path else None,
    "metadata_sha256": file_sha256(KB_METADATA_PATH),
}
KB_PROVENANCE_ID = hashlib.sha256(
    json.dumps(KB_FILE_HASHES, sort_keys=True).encode("utf-8")
).hexdigest()

# Notebook RAG cũ dùng tên trường hash FAISS hơi khác v6. Tính thêm legacy ID
# để nhận ra đúng cùng một KB vật lý, tránh bắt train lại chỉ vì đổi schema metadata.
KB_LEGACY_FILE_HASHES = {
    "embeddings_sha256": file_sha256(KB_EMBEDDING_PATH),
    "mapping_sha256": file_sha256(KB_MAPPING_PATH),
}
legacy_faiss_path = KB_DIR / "text_embeddings_train_restval.faiss"
legacy_index_path = KB_DIR / "faiss_index_train_restval.index"
if legacy_faiss_path.is_file():
    KB_LEGACY_FILE_HASHES["faiss_sha256"] = file_sha256(legacy_faiss_path)
if legacy_index_path.is_file():
    KB_LEGACY_FILE_HASHES["index_sha256"] = file_sha256(legacy_index_path)
if KB_METADATA_PATH.is_file():
    KB_LEGACY_FILE_HASHES["metadata_sha256"] = file_sha256(KB_METADATA_PATH)
KB_LEGACY_PROVENANCE_ID = hashlib.sha256(
    json.dumps(KB_LEGACY_FILE_HASHES, sort_keys=True).encode("utf-8")
).hexdigest()
ACCEPTED_KB_PROVENANCE_IDS = {KB_PROVENANCE_ID, KB_LEGACY_PROVENANCE_ID}

if KB_METADATA_PATH.is_file():
    kb_metadata = load_json(KB_METADATA_PATH)
    assert int(kb_metadata.get("caption_count", 566747)) == 566747
    assert set(kb_metadata.get("included_splits", ["train", "restval"])) <= {"train", "restval"}
    assert int(kb_metadata.get("embedding_dim", 512)) == 512
else:
    kb_metadata = {}

print("=" * 90)
print("KB captions        :", len(kb_captions))
print("KB unique images   :", len(set(kb_image_ids.tolist())))
print("KB splits          :", sorted(set(kb_splits)))
print("FAISS              :", type(kb_index).__name__, kb_index.ntotal, "x", kb_index.d)
print("KB provenance ID   :", KB_PROVENANCE_ID)
print("KB legacy ID       :", KB_LEGACY_PROVENANCE_ID)
print("KB LEAKAGE + ALIGNMENT CONTRACT: PASS")
print("=" * 90)

if MATERIALIZE_KB_EMBEDDINGS_IN_RAM:
    print("Đang copy KB embeddings vào host RAM để training nhanh hơn...")
    kb_embeddings = np.asarray(kb_embeddings, dtype=np.float32).copy()
    assert kb_embeddings.shape == (566747, 512)
    print("KB embeddings đã nằm trong RAM.")

if KB_PROVENANCE_ID != "b9a3a6db4561398f1fa5ce9985524196344c3768fb0607e2ba6953d7fe43ff71":
    raise ValueError("KB files changed from the verified historical L2 protocol.")
DATA_FINGERPRINTS["knowledge_base"] = KB_FILE_HASHES
del kb_captions, kb_splits

# Retrieval source cache is read only. Missing cache is calculated only in RAM.
retrieval_indices, retrieval_scores = {}, {}
# Fingerprint all source cache pairs for a common protocol across selected seeds.
# Seed42 can therefore hash train/val cache files without computing missing train retrieval.
RETRIEVAL_SOURCE_FINGERPRINTS = {}
for split in ("train","val","test"):
    stem=f"{split}_k5_n50_similarity"
    for name in (stem+".npz",stem+"_meta.json"):
        path=SOURCE_CACHE_DIR/name
        RETRIEVAL_SOURCE_FINGERPRINTS[name]=sha256_file(path) if path.is_file() else None
DATA_FINGERPRINTS["historical_retrieval_cache_files"]=RETRIEVAL_SOURCE_FINGERPRINTS
def validate_retrieval(split, rows, scores):
    if rows.shape != scores.shape or rows.shape != (len(image_ids[split]), TOP_K):
        raise ValueError(f"Wrong retrieval shape: {split}")
    if rows.dtype.kind not in "iu" or (rows < 0).any() or (rows >= len(kb_embeddings)).any():
        raise ValueError(f"Invalid KB row IDs: {split}")
    if not np.isfinite(scores).all() or ((scores < -1.001) | (scores > 1.001)).any():
        raise ValueError(f"Invalid cosine scores: {split}")
    if np.any(kb_image_ids[rows] == np.asarray(image_ids[split])[:,None]):
        raise ValueError(f"Retrieval self-leakage: {split}")
    if np.any(np.diff(scores, axis=1) > 1e-5):
        raise ValueError(f"Unsorted retrieval scores: {split}")
    for start in range(0,len(rows),1024):
        r = rows[start:start+1024]
        if any(len(set(x.tolist())) != TOP_K for x in r):
            raise ValueError(f"Duplicate retrieval row: {split}")
        q = np.array(features[split][start:start+1024], dtype=np.float32, copy=True)
        faiss.normalize_L2(q)
        expected_scores = np.einsum("bd,bkd->bk",q,np.asarray(kb_embeddings[r], dtype=np.float32))
        if not np.allclose(scores[start:start+1024], expected_scores, atol=2e-4, rtol=2e-4):
            raise ValueError(f"Retrieval score/embedding alignment mismatch: {split}")

def load_or_retrieve_ram(split):
    stem = f"{split}_k5_n50_similarity"
    cache = SOURCE_CACHE_DIR / (stem + ".npz")
    metadata_path = SOURCE_CACHE_DIR / (stem + "_meta.json")
    if cache.is_file() and metadata_path.is_file():
        before = sha256_file(cache)
        before_meta = sha256_file(metadata_path)
        meta = read_json(metadata_path)
        expected = {"protocol_version":"retrieval_cache_v2", "split":split,
            "num_images":len(image_ids[split]), "image_ids_sha256":hash_id_list(image_ids[split]),
            "feature_sha256":DATA_FINGERPRINTS[f"{split}/{split}_features.npy"],
            "kb_provenance_id":KB_PROVENANCE_ID, "top_n":50, "top_k":5,
            "ranking_mode":"similarity", "deduplicate_source_images":False}
        if any(meta.get(k) != v for k,v in expected.items()):
            raise ValueError(f"Source retrieval cache metadata mismatch; keep files: {cache}")
        with np.load(cache, allow_pickle=False) as data:
            rows = np.array(data["row_indices"], copy=True)
            scores = np.array(data["scores"], dtype=np.float32, copy=True)
        validate_retrieval(split, rows, scores)
        if before != sha256_file(cache) or before_meta != sha256_file(metadata_path):
            raise RuntimeError("Source cache changed during read.")
        print(f"Read verified historical retrieval cache [{split}]: {rows.shape}")
    else:
        if cache.exists() or metadata_path.exists():
            raise FileNotFoundError(f"Incomplete source cache pair; keep source files: {cache}")
        print(f"Retrieving [{split}] in RAM; this may take time. No source cache is written.")
        rows = np.empty((len(image_ids[split]),TOP_K), dtype=np.int64)
        scores = np.empty(rows.shape, dtype=np.float32)
        for start in tqdm(range(0,len(rows),1024), desc=f"RAM FAISS {split}"):
            q=np.array(features[split][start:start+1024],dtype=np.float32,copy=True)
            faiss.normalize_L2(q)
            candidates_s,candidates_r=kb_index.search(q,TOP_N)
            for pos,(candidate_rows,candidate_scores) in enumerate(zip(candidates_r,candidates_s)):
                chosen=[(int(r),float(s)) for r,s in zip(candidate_rows,candidate_scores)
                        if r>=0 and int(kb_image_ids[r]) != int(image_ids[split][start+pos])][:TOP_K]
                if len(chosen)!=TOP_K:
                    raise RuntimeError("Not enough retrieval candidates after self-exclusion.")
                rows[start+pos]=[x[0] for x in chosen]
                scores[start+pos]=[x[1] for x in chosen]
        validate_retrieval(split,rows,scores)
    return rows.astype(np.int64,copy=False), scores

for split in (("test",) if SELECTED_SEED==42 else ("train","val","test")):
    retrieval_indices[split],retrieval_scores[split]=load_or_retrieve_ram(split)
DATA_FINGERPRINTS["test_retrieval_rows"]=hashlib.sha256(retrieval_indices["test"].tobytes()).hexdigest()
DATA_FINGERPRINTS["test_retrieval_scores"]=hashlib.sha256(retrieval_scores["test"].tobytes()).hexdigest()

print("Level2: common fixed Level1 seed42 warm-start; stage seeds vary only fine-tuning.")
print("Historical Level2 seed42 checkpoint:",SEED42_CHECKPOINT)
print("All source checkpoints, caches, KB, vocab and annotations are read only.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: 113,287 ảnh • 566,747 captions • (113287, 512)
val: 5,000 ảnh • 25,010 captions • (5000, 512)
test: 5,000 ảnh • 25,010 captions • (5000, 512)
Vocab=16,214 • GPU=Tesla T4 • annotations=/content/drive/MyDrive/DataKhoaLuan/Data/COCO_2014/annotations_trainval2014/annotations
Seed được chọn: 44
Vùng đầu ra mới (sẽ xác định sau khi kiểm tra protocol): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/multiseed_beam3_v1


Load KB mapping:   0%|          | 0/566747 [00:00<?, ?it/s]

KB captions        : 566747
KB unique images   : 113287
KB splits          : ['restval', 'train']
FAISS              : IndexFlatIP 566747 x 512
KB provenance ID   : b9a3a6db4561398f1fa5ce9985524196344c3768fb0607e2ba6953d7fe43ff71
KB legacy ID       : ad5ae232137e16a4c5c5c545d38fc29ee1d7a1e67c7ed5e448ba385a46f4140f
KB LEAKAGE + ALIGNMENT CONTRACT: PASS
Đang copy KB embeddings vào host RAM để training nhanh hơn...
KB embeddings đã nằm trong RAM.
Retrieving [train] in RAM; this may take time. No source cache is written.


RAM FAISS train:   0%|          | 0/111 [00:00<?, ?it/s]

Retrieving [val] in RAM; this may take time. No source cache is written.


RAM FAISS val:   0%|          | 0/5 [00:00<?, ?it/s]

Read verified historical retrieval cache [test]: (5000, 5)
Level2: common fixed Level1 seed42 warm-start; stage seeds vary only fine-tuning.
Historical Level2 seed42 checkpoint: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/rag_outputs/R1_K5_N50_A0.5_seed42_L4/checkpoint_best.pt
All source checkpoints, caches, KB, vocab and annotations are read only.


In [7]:
#@title CELL 3 — Model R1, Beam k3, protocol và lưu an toàn
class BaselineCaptioner(nn.Module):
    def __init__(self, clip_dim, vocab_size, d_model, nhead, num_layers, dropout, max_length):
        super().__init__()
        self.d_model = d_model
        self.max_length = max_length
        self.feature_projection = nn.Linear(clip_dim, d_model)
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_length, d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            norm_first=False,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, vocab_size)

    @staticmethod
    def causal_mask(length, device):
        return torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=device), diagonal=1
        )

    def forward(self, image_features, input_ids):
        batch_size, seq_len = input_ids.shape
        if seq_len > self.max_length:
            raise ValueError(f"seq_len={seq_len} > max_length={self.max_length}")
        memory = self.feature_projection(image_features).unsqueeze(1)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        target = self.token_embedding(input_ids) + self.position_embedding(positions)
        decoded = self.decoder(
            target,
            memory,
            tgt_mask=self.causal_mask(seq_len, input_ids.device),
            tgt_key_padding_mask=input_ids.eq(PAD_ID),
        )
        return self.output_projection(decoded)


class RAGBaselineLevel2(nn.Module):
    ARCHITECTURE_VERSION = "rag_weighted_sum_v1"

    def __init__(self, decoder, alpha=0.5, temperature=1.0):
        super().__init__()
        self.decoder = decoder
        self.alpha = float(alpha)
        self.temperature = float(temperature)
        # Giữ layer này để strict-load tương thích checkpoint R1 cũ.
        self.concat_proj = nn.Linear(512 * 2, 512)

    def fuse(self, visual, knowledge_features=None, retrieval_scores=None, mode="weighted_sum"):
        visual = F.normalize(visual, p=2, dim=-1)
        if mode == "visual_only":
            return visual
        if knowledge_features is None or retrieval_scores is None:
            raise ValueError("weighted_sum cần knowledge_features và retrieval_scores")
        weights = torch.softmax(retrieval_scores / self.temperature, dim=-1)
        knowledge = torch.bmm(weights.unsqueeze(1), knowledge_features).squeeze(1)
        knowledge = F.normalize(knowledge, p=2, dim=-1)
        if mode == "knowledge_only":
            return knowledge
        if mode == "concat_linear":
            return F.normalize(self.concat_proj(torch.cat([visual, knowledge], dim=-1)), p=2, dim=-1)
        if mode != "weighted_sum":
            raise ValueError(f"Fusion mode không hợp lệ: {mode}")
        return F.normalize(self.alpha * visual + (1.0 - self.alpha) * knowledge, p=2, dim=-1)

    def forward(self, visual, knowledge_features, retrieval_scores, input_ids, fusion_mode="weighted_sum"):
        fused = self.fuse(visual, knowledge_features, retrieval_scores, mode=fusion_mode)
        return self.decoder(fused, input_ids)


def new_model():
    decoder=BaselineCaptioner(512,VOCAB_SIZE,256,8,3,0.1,MAX_LEN)
    return RAGBaselineLevel2(decoder,FUSION_ALPHA,RETRIEVAL_TEMPERATURE).to(DEVICE)

def model_initialized_from_fixed_upstream():
    model=new_model()
    state=dict(UPSTREAM_PAYLOAD["model_state_dict"])
    if "feature_proj.weight" in state and "feature_projection.weight" not in state:
        state["feature_projection.weight"]=state.pop("feature_proj.weight")
        state["feature_projection.bias"]=state.pop("feature_proj.bias")
    model.decoder.load_state_dict(state,strict=True)
    return model

@torch.inference_mode()
def beam_search_decode(model, image_feature, beam_size=3, max_len=30, alpha=0.6):
    """max_len gồm BOS; mỗi bước xếp hạng cả beam EOS và beam đang mở rộng."""
    if beam_size < 1 or not 2 <= max_len <= model.position_embedding.num_embeddings:
        raise ValueError("beam_size >= 1 và 2 <= max_len <= 30.")
    model.eval()
    device = next(model.parameters()).device
    image_feature = torch.as_tensor(image_feature, dtype=torch.float32, device=device)
    if image_feature.ndim == 2 and image_feature.shape[0] == 1:
        image_feature = image_feature[0]
    if image_feature.shape != (512,):
        raise ValueError(f"Cần một feature 512 chiều, nhận {image_feature.shape}.")
    beams = [([BOS_ID], 0.0)]
    def normalized(beam):
        tokens, score = beam
        return score / (((5.0 + len(tokens) - 1) / 6.0) ** alpha)
    for _ in range(max_len - 1):
        active = [(tokens, score) for tokens, score in beams if tokens[-1] != EOS_ID]
        if not active:
            break
        candidates = [(tokens, score) for tokens, score in beams if tokens[-1] == EOS_ID]
        inputs = torch.tensor([tokens for tokens, _ in active], device=device)
        memory = image_feature.unsqueeze(0).expand(len(active), -1)
        logits = model(memory, inputs)[:, -1, :].float()
        if not torch.isfinite(logits).all():
            raise RuntimeError("Model sinh logits không hữu hạn; dừng để kiểm tra checkpoint.")
        logits[:, [PAD_ID, BOS_ID]] = -torch.inf
        log_prob = torch.log_softmax(logits, dim=-1)
        top_scores, top_ids = log_prob.topk(min(beam_size, VOCAB_SIZE - 2), dim=-1)
        # Một lần chuyển GPU -> CPU cho toàn bộ beam ở bước này.
        for (tokens, score), values, indices in zip(
                active, top_scores.tolist(), top_ids.tolist()):
            candidates.extend((tokens + [int(token)], score + float(value))
                              for value, token in zip(values, indices))
        beams = sorted(candidates, key=normalized, reverse=True)[:beam_size]
        if all(tokens[-1] == EOS_ID for tokens, _ in beams):
            break
    best_tokens, _ = max(beams, key=normalized)
    return " ".join(itos[t] for t in best_tokens if t not in (PAD_ID, BOS_ID, EOS_ID))

ARCHITECTURE = dict(name="rag_weighted_sum_v1",clip_dim=512,d_model=256,nhead=8,
    num_layers=3,dim_feedforward=1024,dropout=0.1,max_length=30,vocab_size=VOCAB_SIZE,
    fusion_alpha=0.5,temperature=1.0,top_k=5,top_n=50,ranking="similarity",
    deduplicate_source_images=False,exclude_query_image=True)
TRAIN_CONFIG = dict(max_epochs=30,patience=3,min_delta=1e-4,
    learning_rate=3e-4,weight_decay=1e-4,batch_size=32,gradient_clip=1.0,
    loss_aggregation="nonpad_token_weighted",best_selection="min_delta_like_original",
    precision="bf16_autocast" if USE_BF16 else "float32",tf32_enabled=USE_TF32,
    fused_adamw=USE_FUSED_ADAMW,
    dynamic_padding=True,num_workers=0,upstream_fixed_seed=42,
    policy="conditional_joint_finetune_from_historical_Level1_seed42")
DECODE_CONFIG = dict(method="beam_search",beam_size=3,alpha=0.6,max_len=30,
    max_len_includes_bos=True,mask_pad_bos=True,implementation="normalized_candidates_v1",
    precision="float32",tf32_enabled=False)
PROTOCOL = dict(level=2,architecture=ARCHITECTURE,train=TRAIN_CONFIG,
    decode=DECODE_CONFIG,data=DATA_FINGERPRINTS,tokenizer=TOKEN_PATTERN.pattern,
    evaluator="pycocoevalcap==1.2/PTB/Java11",storage_schema="per_seed_guarded_v2",
    experiment_tag=EXPERIMENT_TAG,seed42_checkpoint_sha256=EXPECTED_SEED42_SHA256,
    inference_caveat="historical seed42 reused; seeds44/46 vary conditional stage only",
    recipe_provenance={"budget_lr_batch_patience":"historical Drive config_R1.json",
        "loss_padding_min_delta":"available v6 source; not proven identical to historical training",
        "historical_hardware":"NVIDIA L4 / bf16 / TF32",
        "claim":"conditional stage reruns with declared historical recipe gaps"},
    implementation_version="level2_multiseed_beam3_v1")
PROTOCOL_HASH = canonical_hash(PROTOCOL)
METRIC_NAMES=["Bleu_1","Bleu_2","Bleu_3","Bleu_4","METEOR","ROUGE_L","CIDEr","SPICE"]

# Mọi đường dẫn ghi đều bị giới hạn vào RUN_DIR; input gốc chỉ được đọc.
import uuid
from contextlib import contextmanager
RUN_DIR = OUTPUT_ROOT / ("run_" + PROTOCOL_HASH[:16])
RESULT_DIR = RUN_DIR
_OBSERVED_FILES = {}
_ACTIVE_LOCKS = {}

def output_path(path):
    path = Path(path).resolve()
    try:
        path.relative_to(RUN_DIR.resolve())
    except ValueError as exc:
        raise PermissionError(f"Từ chối ghi ngoài vùng kết quả mới: {path}") from exc
    return path

def owned_meta(seed, role):
    return {"protocol_hash": PROTOCOL_HASH, "storage_schema": 2,
            "seed": seed, "role": role}

def verify_owner(payload, seed, role):
    expected = owned_meta(seed, role)
    meta = payload.get("_meta", {}) if isinstance(payload, dict) else {}
    if any(meta.get(k) != v for k, v in expected.items()):
        raise ValueError(f"File khác seed/protocol/định dạng ({role}); giữ nguyên và dừng.")

def remember_file(path, before):
    path = output_path(path)
    after = sha256_file(path)
    if before != after:
        raise RuntimeError(f"File đang bị phiên khác thay đổi: {path.name}")
    _OBSERVED_FILES[str(path)] = after

def read_owned_json(path, seed, role):
    path = output_path(path)
    if not path.exists():
        _OBSERVED_FILES[str(path)] = None
        return None
    if not path.is_file():
        raise FileExistsError(f"Đường dẫn đã là thư mục: {path}")
    before = sha256_file(path)
    payload = read_json(path)  # JSON lỗi: dừng, không tự ghi đè.
    verify_owner(payload, seed, role)
    remember_file(path, before)
    return payload

def seed_dir(seed, create=False):
    if seed not in (42, 44, 46):
        raise ValueError("Chỉ có seed 42, 44, 46.")
    path = output_path(RUN_DIR / f"seed_{seed}")
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path

def check_lock(scope):
    """Verify an active lock; tolerate short Drive read glitches, never ownership changes."""
    lock = _ACTIVE_LOCKS.get(scope)
    if lock is None:
        raise RuntimeError(f"Thiếu khóa ghi cho {scope}.")
    path, token = lock
    last_error = None
    for attempt in range(5):
        try:
            current = read_json(path)
            if not isinstance(current, dict):
                raise ValueError("Lock JSON không hợp lệ")
            verify_owner(current, scope, "active_run")
            if current.get("token") != token:
                raise RuntimeError("Quyền ghi đã thay đổi; dừng để bảo vệ dữ liệu.")
            return
        except RuntimeError:
            raise
        except (OSError, ValueError, TypeError) as exc:
            last_error = exc
            if attempt < 4:
                time.sleep(0.5 * (attempt + 1))
    raise RuntimeError("Không đọc được khóa sau 5 lần thử; dừng ghi an toàn.") from last_error

@contextmanager
def writing_scope(scope):
    global OLD_SESSION_STOPPED
    # Một phiên đang chạy cho mỗi seed. Khóa trên Drive là lớp phòng vệ,
    # không thay thế quy tắc không chạy hai Colab VM cùng seed.
    folder = RUN_DIR if scope == "summary" else seed_dir(scope, create=True)
    lock_path = output_path(folder / ".active_run.json")
    if scope in _ACTIVE_LOCKS:
        raise RuntimeError(f"Seed/phần tổng hợp {scope} đang chạy trong phiên này.")
    if lock_path.exists():
        old_hash = sha256_file(lock_path)
        try:
            old = read_json(lock_path)
        except (OSError, ValueError):
            old = None
        if not OLD_SESSION_STOPPED:
            raise RuntimeError(
                f"Còn phiên/khóa cũ của {scope}. Nếu phiên cũ đã DỪNG HẲN, bật "
                "OLD_SESSION_STOPPED ở Cell 2, rồi chạy lại. Không chạy hai phiên cùng seed.")
        if old is not None:
            verify_owner(old, scope, "active_run")
        elif lock_path.stat().st_size > 4096:
            raise RuntimeError("File khóa hỏng có kích thước bất thường; giữ nguyên để kiểm tra.")
        if sha256_file(lock_path) != old_hash:
            raise RuntimeError("Khóa vừa đổi; không thu hồi.")
        lock_path.unlink()
        OLD_SESSION_STOPPED = False  # Thu hồi một lần, tránh vô tình lấy khóa khác.
    token = uuid.uuid4().hex
    record = {"_meta": owned_meta(scope, "active_run"), "token": token,
              "started_unix": time.time()}
    with lock_path.open("x", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False)
    _ACTIVE_LOCKS[scope] = (lock_path, token)
    try:
        check_lock(scope)
        yield
    finally:
        try:
            if lock_path.is_file():
                try:
                    existing = read_json(lock_path)
                    if isinstance(existing, dict) and existing.get("token") == token:
                        lock_path.unlink()
                except (OSError, ValueError) as exc:
                    print(f"Cảnh báo: chưa thể dọn khóa {lock_path}: {exc}")
        finally:
            _ACTIVE_LOCKS.pop(scope, None)

def required_scope(path):
    relative = output_path(path).relative_to(RUN_DIR.resolve())
    if len(relative.parts) > 1 and relative.parts[0].startswith("seed_"):
        return int(relative.parts[0].split("_")[1])
    return "summary"

def commit_file(path, write_temp, mutable=False):
    path = output_path(path)
    scope = required_scope(path)
    check_lock(scope)
    key = str(path)
    if key not in _OBSERVED_FILES:
        if path.exists():
            raise FileExistsError(f"Chưa xác minh file có sẵn; KHÔNG ghi đè: {path}")
        _OBSERVED_FILES[key] = None
    expected = _OBSERVED_FILES[key]
    current = sha256_file(path) if path.exists() else None
    if current != expected:
        raise RuntimeError(f"File đã đổi sau khi đọc; KHÔNG ghi đè: {path}")
    tmp = output_path(path.with_name("." + path.name + "." + uuid.uuid4().hex + ".tmp"))
    try:
        write_temp(tmp)
        new_hash = sha256_file(tmp)
        if current == new_hash:
            return False
        if current is not None and not mutable:
            raise FileExistsError(f"Kết quả đã có và khác nội dung; giữ nguyên: {path}")
        check_lock(scope)
        if (sha256_file(path) if path.exists() else None) != expected:
            raise RuntimeError(f"File vừa đổi; KHÔNG ghi đè: {path}")
        # Chỉ thay trạng thái đang chạy của đúng experiment/seed đã xác minh.
        # Không di chuyển hay đổi tên bất kỳ dữ liệu/checkpoint nguồn nào.
        os.replace(tmp, path)
        _OBSERVED_FILES[key] = new_hash
        if sha256_file(path) != new_hash:
            raise IOError(f"Readback sau lưu không khớp: {path}")
        return True
    finally:
        # Chỉ xóa file tạm UUID của chính lần ghi này.
        tmp.unlink(missing_ok=True)

def save_json(value, path, mutable=False):
    def write_temp(tmp):
        with tmp.open("x", encoding="utf-8") as f:
            json.dump(value, f, ensure_ascii=False, sort_keys=True,
                      indent=2, allow_nan=False)
            f.flush()
            os.fsync(f.fileno())
    return commit_file(path, write_temp, mutable=mutable)

def initialize_experiment():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = RUN_DIR / "experiment.json"
    manifest = {"_meta": owned_meta(None, "experiment"),
                "protocol": PROTOCOL, "seeds": [42, 44, 46]}
    def validate_manifest():
        last_error = None
        for attempt in range(6):
            try:
                previous = read_owned_json(manifest_path, None, "experiment")
                if previous is not None:
                    if previous != manifest:
                        raise ValueError("Manifest khác cấu hình; không dùng chung thư mục.")
                    return
            except (OSError, json.JSONDecodeError) as exc:
                last_error = exc
            if attempt < 5:
                time.sleep(0.2)
        raise RuntimeError(
            "Manifest thiếu/hỏng hoặc chưa ghi xong; mọi file được giữ nguyên. "
            "Thử chạy lại Cell 3. Nếu vẫn lỗi, đổi EXPERIMENT_TAG ở Cell 2 để "
            "tạo thí nghiệm mới, không xóa/ghi đè thư mục cũ.") from last_error

    if manifest_path.exists():
        validate_manifest()
    elif any(RUN_DIR.iterdir()):
        # Seed khác có thể vừa khởi tạo manifest giữa exists() và iterdir().
        validate_manifest()
    else:
        try:
            with manifest_path.open("x", encoding="utf-8") as f:
                json.dump(manifest, f, ensure_ascii=False, sort_keys=True, indent=2)
                f.flush()
                os.fsync(f.fileno())
        except FileExistsError:
            validate_manifest()
    print("Kết quả mới:", RUN_DIR)
    print(f"Seed đang chọn: {SELECTED_SEED} →", seed_dir(SELECTED_SEED))
    print("Checkpoint/vocab/features/annotations gốc: CHỈ ĐỌC.")

def load_checkpoint(path,expected_seed=None):
    path=Path(path)
    before=sha256_file(path)
    if expected_seed is None and (path.resolve()!=SEED42_CHECKPOINT.resolve()
            or before!=EXPECTED_SEED42_SHA256):
        raise ValueError("Historical seed42 checkpoint path/hash mismatch.")
    if expected_seed is None:
        # Historical R1 stores NumPy uint32 RNG state. Allow only these concrete
        # NumPy types after verifying the exact trusted source checkpoint hash.
        core=getattr(np,"_core",None)
        if core is None:
            core=np.core
        reconstruct=core.multiarray._reconstruct
        scalar=core.multiarray.scalar
        allowed=[np.ndarray,np.dtype,type(np.dtype("uint32")),
            type(np.dtype("float32")),type(np.dtype("float64")),
            (reconstruct,"numpy._core.multiarray._reconstruct"),
            (reconstruct,"numpy.core.multiarray._reconstruct"),
            (scalar,"numpy._core.multiarray.scalar"),
            (scalar,"numpy.core.multiarray.scalar")]
        with torch.serialization.safe_globals(allowed):
            checkpoint=torch.load(path,map_location="cpu",weights_only=True)
    else:
        checkpoint=torch.load(path,map_location="cpu",weights_only=True)
    if not isinstance(checkpoint,dict) or "model_state_dict" not in checkpoint:
        raise ValueError(f"Invalid checkpoint: {path}")
    if expected_seed is not None:
        output_path(path)
        verify_owner(checkpoint,expected_seed,"checkpoint")
        if checkpoint.get("seed")!=expected_seed or checkpoint.get("protocol_hash")!=PROTOCOL_HASH:
            raise ValueError("Wrong checkpoint seed/protocol; keep file.")
        if checkpoint.get("stoi")!=stoi or checkpoint.get("itos")!=itos:
            raise ValueError("Checkpoint vocab mismatch.")
        remember_file(path,before)
    else:
        expected={"top_n":50,"top_k":5,"alpha":0.5,
            "b0_checkpoint_hash":EXPECTED_UPSTREAM_SHA256,
            "vocab_hash":DATA_FINGERPRINTS["vocab"],
            "baseline_config_hash":DATA_FINGERPRINTS["baseline_config"]}
        if any(checkpoint.get(k)!=v for k,v in expected.items()):
            raise ValueError("Historical R1 checkpoint training/data provenance mismatch.")
        if checkpoint.get("kb_provenance_id") not in ACCEPTED_KB_PROVENANCE_IDS:
            raise ValueError("Historical R1 KB provenance mismatch.")
        for k,v in {"retrieval_temperature":1.0,"ranking_mode":"similarity",
                    "deduplicate_source_images":False,"seed":42}.items():
            if k in checkpoint and checkpoint[k]!=v:
                raise ValueError(f"Historical R1 mismatch: {k}")
        if sha256_file(path)!=before:
            raise RuntimeError("Historical checkpoint changed during read.")
    return checkpoint

def model_from_checkpoint(path,expected_seed=None):
    # Same float32 inference arithmetic policy as the new Level1 evaluator.
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    checkpoint=load_checkpoint(path,expected_seed)
    model=new_model()
    model.load_state_dict(checkpoint["model_state_dict"],strict=True)
    return model.eval()

@torch.inference_mode()
def fused_feature_for_test(model,index,visual):
    rows=retrieval_indices["test"][index]
    knowledge=torch.from_numpy(np.array(kb_embeddings[rows],dtype=np.float32,copy=True)).to(DEVICE)
    scores=torch.from_numpy(retrieval_scores["test"][index].copy()).to(DEVICE)
    fused=model.fuse(visual.to(DEVICE).reshape(1,512),knowledge.unsqueeze(0),scores.unsqueeze(0))
    return fused[0]

# Strict-check historical full wrapper and upstream decoder before output creation.
_historical_check=model_from_checkpoint(SEED42_CHECKPOINT)
_upstream_check=model_initialized_from_fixed_upstream()
assert sum(p.numel() for p in _historical_check.parameters())==12141910
assert _historical_check.decoder(torch.zeros(1,512,device=DEVICE),
    torch.tensor([[BOS_ID]],device=DEVICE)).shape==(1,1,VOCAB_SIZE)
del _historical_check,_upstream_check
gc.collect()
torch.cuda.empty_cache()
print(f"Level2 model/provenance strict-load PASS • protocol {PROTOCOL_HASH[:12]}")
initialize_experiment()


Level2 model/provenance strict-load PASS • protocol 3aace7f5829a
Kết quả mới: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/multiseed_beam3_v1/run_3aace7f5829acedd
Seed đang chọn: 44 → /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level2/multiseed_beam3_v1/run_3aace7f5829acedd/seed_44
Checkpoint/vocab/features/annotations gốc: CHỈ ĐỌC.


In [8]:
#@title CELL 4 — Hàm đánh giá có resume; chỉ chạy seed 42 khi được chọn
def prediction_prefix_valid(predictions, complete=False):
    if not isinstance(predictions, list) or len(predictions) > len(image_ids["test"]):
        return False
    try:
        ids = [int(row["image_id"]) for row in predictions]
        valid = (ids == image_ids["test"][:len(ids)]
                 and len(ids) == len(set(ids))
                 and all(isinstance(row["caption"], str) for row in predictions))
        return valid and (not complete or len(ids) == 5000)
    except (KeyError, TypeError, ValueError):
        return False

def valid_metrics(metrics):
    return all(name in metrics and isinstance(metrics[name], (int, float))
               and math.isfinite(float(metrics[name])) for name in METRIC_NAMES)

_TOKENIZED_REFERENCES = None
def score_predictions(predictions):
    global _TOKENIZED_REFERENCES
    if not prediction_prefix_valid(predictions, complete=True):
        raise ValueError("Predictions phải phủ đúng 5.000 test IDs theo thứ tự cache.")
    if _TOKENIZED_REFERENCES is None:
        print("PTB tokenize toàn bộ 25.010 reference captions...")
        _TOKENIZED_REFERENCES = PTBTokenizer().tokenize({
            i: [{"caption": caption} for caption in references[i]]
            for i in image_ids["test"]})
    prediction_by_id = {int(p["image_id"]): p["caption"] for p in predictions}
    res = PTBTokenizer().tokenize({
        i: [{"caption": prediction_by_id[i]}] for i in image_ids["test"]})
    metrics = {}
    scorers = [(lambda: Bleu(4), METRIC_NAMES[:4]),
               (ManagedMeteor, ["METEOR"]), (Rouge, ["ROUGE_L"]),
               (Cider, ["CIDEr"]), (Spice, ["SPICE"])]
    for factory, names in scorers:
        print("Đang chấm:", ", ".join(names), flush=True)
        scorer = factory()
        try:
            score, _ = scorer.compute_score(_TOKENIZED_REFERENCES, res)
            for name, value in zip(names, score if len(names) > 1 else [score]):
                value = float(value)
                if not math.isfinite(value):
                    raise ValueError(f"{name} không hữu hạn.")
                metrics[name] = value
        except Exception as exc:
            raise RuntimeError(f"Chấm {names} thất bại; giữ predictions để chấm lại: {exc}") from exc
        finally:
            if isinstance(scorer, ManagedMeteor):
                scorer.close()
            del scorer
            gc.collect()
    if not valid_metrics(metrics):
        raise RuntimeError("Chưa đủ 8 chỉ số hợp lệ; không ghi điểm giả.")
    return metrics

def load_prediction_packet(path, seed, signature):
    packet = read_owned_json(path, seed, "predictions")
    if packet is None:
        return None
    meta = packet["_meta"]
    predictions = packet.get("predictions")
    if meta.get("signature") != signature:
        raise ValueError("Predictions khác checkpoint/cấu hình; GIỮ NGUYÊN, không ghi đè.")
    complete = meta.get("complete")
    if not isinstance(complete, bool) or not prediction_prefix_valid(predictions, complete):
        raise ValueError("Predictions hỏng/trùng/sai thứ tự ID; giữ nguyên và dừng.")
    if meta.get("predictions_hash") != canonical_hash(predictions):
        raise ValueError("Nội dung predictions đã đổi; giữ nguyên và dừng.")
    return packet

def evaluate_seed(seed, checkpoint_path):
    check_lock(seed)
    checkpoint_path = Path(checkpoint_path)
    checkpoint_hash = sha256_file(checkpoint_path)
    checked = load_checkpoint(checkpoint_path, None if seed == 42 else seed)
    if seed != 42 and not checked.get("training_complete", False):
        raise RuntimeError("Checkpoint chưa hoàn tất train; không đánh giá như kết quả cuối.")
    checked_epoch=checked.get("best_epoch",checked.get("epoch",-1))
    del checked
    metrics_path = seed_dir(seed) / f"metrics_seed{seed}.json"
    predictions_path = seed_dir(seed) / f"predictions_seed{seed}.json"
    signature = canonical_hash({"seed": seed, "checkpoint_sha256": checkpoint_hash,
                               "protocol_hash": PROTOCOL_HASH})
    packet = load_prediction_packet(predictions_path, seed, signature)
    previous = read_owned_json(metrics_path, seed, "metrics")
    if previous is not None:
        if (previous["_meta"].get("signature") != signature or not valid_metrics(previous)
                or previous["_meta"].get("status") != "complete"
                or packet is None or not packet["_meta"]["complete"]
                or previous["_meta"].get("predictions_sha256") != sha256_file(predictions_path)):
            raise ValueError("Có metrics nhưng không xác minh được; GIỮ NGUYÊN và dừng.")
        if sha256_file(checkpoint_path) != checkpoint_hash:
            raise RuntimeError("Checkpoint vừa đổi trong lúc kiểm tra; giữ nguyên kết quả.")
        print(f"Seed {seed}: đã đủ kết quả hợp lệ; bỏ qua, không sửa file.")
        return previous

    predictions = [] if packet is None else list(packet["predictions"])
    start = time.perf_counter()
    def persist_predictions(complete):
        nonlocal packet
        if packet is not None and packet["_meta"]["complete"]:
            raise FileExistsError("Predictions đã hoàn tất; không thay thế.")
        if not prediction_prefix_valid(predictions, complete):
            raise ValueError("Không ghi predictions sai coverage.")
        updated = {
            "_meta": {**owned_meta(seed, "predictions"), "signature": signature,
                "checkpoint_sha256": checkpoint_hash, "complete": complete,
                "num_images": len(predictions),
                "predictions_hash": canonical_hash(predictions),
                "format": "envelope; COCO rows are in the predictions key"},
            "predictions": predictions,
        }
        save_json(updated, predictions_path, mutable=packet is not None)
        # Copy để trạng thái đã lưu không đổi theo list đang append.
        packet = {"_meta": dict(updated["_meta"]), "predictions": list(predictions)}

    if packet is None or not packet["_meta"]["complete"]:
        print(f"Seed {seed}: tiếp tục Beam Search từ ảnh {len(predictions) + 1}/5000.")
        model = model_from_checkpoint(checkpoint_path, None if seed == 42 else seed)
        if sha256_file(checkpoint_path) != checkpoint_hash:
            del model
            raise RuntimeError("Checkpoint đổi trong lúc nạp; dừng.")
        last_saved_count = len(predictions)
        try:
            with tqdm(total=5000, initial=len(predictions),
                      desc=f"Seed {seed} • Beam k=3", unit="ảnh") as progress:
                for index, (ids, batch_features) in enumerate(test_loader):
                    if index < len(predictions):
                        continue
                    assert int(ids[0]) == image_ids["test"][index]
                    fused=fused_feature_for_test(model,index,batch_features[0])
                    caption = beam_search_decode(model.decoder, fused,
                        beam_size=3, max_len=MAX_LEN, alpha=0.6)
                    predictions.append({"image_id": int(ids[0]), "caption": caption})
                    progress.update(1)
                    if len(predictions) % 100 == 0 and len(predictions) < 5000:
                        persist_predictions(complete=False)
                        last_saved_count = len(predictions)
            if sha256_file(checkpoint_path) != checkpoint_hash:
                raise RuntimeError("Checkpoint đã đổi trong khi decode; không chấm điểm lẫn phiên.")
            persist_predictions(complete=True)
        except BaseException:
            if len(predictions) > last_saved_count and len(predictions) < 5000:
                try:
                    persist_predictions(complete=False)
                except Exception as save_error:
                    print("Không lưu thêm được tiến độ; bản đã lưu trước vẫn giữ:", save_error)
            raise
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    else:
        print(f"Seed {seed}: đã có 5.000 predictions; chỉ chạy phần chấm điểm.")

    if sha256_file(checkpoint_path) != checkpoint_hash:
        raise RuntimeError("Checkpoint đã đổi; không chấm/lưu kết quả cuối.")
    prediction_hash = sha256_file(predictions_path)
    metrics = score_predictions(predictions)
    if (sha256_file(checkpoint_path) != checkpoint_hash
            or sha256_file(predictions_path) != prediction_hash):
        raise RuntimeError("Input đánh giá đã đổi trong lúc chấm; không ghi metrics.")
    metrics["_meta"] = {
        **owned_meta(seed, "metrics"), "status": "complete", "signature": signature,
        "split": "MS COCO 2014 Karpathy Test", "num_images": 5000,
        "num_references": sum(map(len, references.values())),
        "checkpoint": str(checkpoint_path), "checkpoint_sha256": checkpoint_hash,
        "predictions_sha256": prediction_hash, "decoding": DECODE_CONFIG,
        "metric_scale": "raw pycocoevalcap scores, no multiplication by 100",
        "elapsed_seconds_this_session": time.perf_counter() - start,
        "checkpoint_best_epoch": int(checked_epoch),
        "conditional_upstream_seed": 42,
        "training_policy": "historical_fixed_seed42" if seed==42 else TRAIN_CONFIG["policy"],
        "environment": {"python": platform.python_version(), "torch": str(torch.__version__),
            "numpy": np.__version__, "pycocoevalcap": metadata.version("pycocoevalcap"),
            "gpu": torch.cuda.get_device_name(), "java": "11"},
    }
    save_json(metrics, metrics_path)  # Immutable final: không cho thay file khác.
    print(f"Seed {seed} hoàn tất:", {k: round(metrics[k], 6) for k in METRIC_NAMES})
    return metrics

if SELECTED_SEED == 42:
    with writing_scope(42):
        metrics_seed42 = evaluate_seed(42, SEED42_CHECKPOINT)
else:
    print(f"Đang chọn seed {SELECTED_SEED}; Cell 4 chỉ khai báo hàm, không chạy seed 42.")


Đang chọn seed 44; Cell 4 chỉ khai báo hàm, không chạy seed 42.


In [9]:
#@title CELL 5 — Train mới / chạy tiếp seed 44, 46 rồi đánh giá Beam Search
def save_checkpoint(checkpoint, path):
    verify_owner(checkpoint, checkpoint["seed"], "checkpoint")
    def write_temp(tmp):
        with tmp.open("xb") as f:
            torch.save(checkpoint, f)
            f.flush()
            os.fsync(f.fileno())
    return commit_file(path, write_temp, mutable=True)

def capture_rng(generator):
    np_state = np.random.get_state()
    return {
        "python": random.getstate(), "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        "numpy": {"name": np_state[0], "keys": np_state[1].tolist(),
                  "pos": int(np_state[2]), "has_gauss": int(np_state[3]),
                  "cached_gaussian": float(np_state[4])},
        "loader": generator.get_state(),
    }

def restore_rng(state, generator):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch_cpu"])
    if torch.cuda.is_available():
        if len(state["torch_cuda"]) != torch.cuda.device_count():
            raise RuntimeError("Số GPU đã đổi; không thể khôi phục RNG chính xác.")
        torch.cuda.set_rng_state_all(state["torch_cuda"])
    n = state["numpy"]
    np.random.set_state((n["name"], np.asarray(n["keys"], dtype=np.uint32),
                         n["pos"], n["has_gauss"], n["cached_gaussian"]))
    generator.set_state(state["loader"])

def run_epoch(model, loader, criterion, epoch, seed, optimizer=None, split="train"):
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32
    training = optimizer is not None
    model.train(training)
    total_loss, total_tokens = 0.0, 0
    with torch.set_grad_enabled(training):
        for visual, tokens, positions in tqdm(loader, desc=f"Seed{seed} epoch{epoch:02d} {split}", leave=False):
            visual = visual.to(DEVICE, non_blocking=True)
            tokens = tokens.to(DEVICE, non_blocking=True)
            pos = positions.numpy()
            rows = retrieval_indices[split][pos]
            knowledge = torch.from_numpy(np.array(kb_embeddings[rows], dtype=np.float32, copy=True)).to(DEVICE)
            scores = torch.from_numpy(retrieval_scores[split][pos].copy()).to(DEVICE)
            inputs, targets = tokens[:, :-1], tokens[:, 1:]
            count = int(targets.ne(PAD_ID).sum().item())
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=USE_BF16):
                logits = model(visual, knowledge, scores, inputs)
                loss_sum = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
                loss = loss_sum / max(count, 1)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss: seed{seed}, epoch{epoch}, split{split}")
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
                optimizer.step()
            total_loss += float(loss_sum.detach().item())
            total_tokens += count
    if not total_tokens:
        raise RuntimeError("Empty caption DataLoader.")
    return total_loss / total_tokens

def train_seed(seed):
    seed_everything(seed)
    check_lock(seed)
    if seed not in (44, 46):
        raise ValueError("Không train lại seed 42.")
    best_path = seed_dir(seed) / f"checkpoint_best_seed{seed}.pt"
    last_path = seed_dir(seed) / f"checkpoint_last_seed{seed}.pt"
    metrics_path = seed_dir(seed) / f"metrics_seed{seed}.json"

    model = model_initialized_from_fixed_upstream()
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, reduction="sum")
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
        fused=USE_FUSED_ADAMW,
    )

    epoch_start = 1
    best_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    # Hỗ trợ Resume tự động nếu tiến trình bị ngắt giữa chừng
    if last_path.is_file():
        checkpoint = load_checkpoint(last_path, seed)
        if checkpoint.get("training_complete", False):
            print(f"Seed {seed}: Huấn luyện đã hoàn thành từ trước.")
            del model, criterion, optimizer, checkpoint
            gc.collect()
            torch.cuda.empty_cache()
            return
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        epoch_start = checkpoint["epoch"] + 1
        best_loss = checkpoint["best_loss"]
        best_epoch = checkpoint["best_epoch"]
        patience_counter = checkpoint["patience_counter"]
        restore_rng(checkpoint["rng_state"], train_generator)
        print(f"Seed {seed}: Khôi phục thành công từ epoch {checkpoint['epoch']} (best_val_loss={best_loss:.6f})")
        del checkpoint

    for epoch in range(epoch_start, TRAIN_CONFIG["max_epochs"] + 1):
        check_lock(seed)
        loader, _ = make_train_loader(seed)
        train_loss = run_epoch(model, loader, criterion, epoch, seed, optimizer, "train")
        val_loss = run_epoch(model, val_loader, criterion, epoch, seed, None, "val")
        del loader

        is_best = False
        # Quy tắc min_delta tương tự như bản gốc
        if val_loss < best_loss - TRAIN_CONFIG["min_delta"]:
            best_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            is_best = True
        else:
            patience_counter += 1

        rng = capture_rng(train_generator)
        state = {
            "_meta": owned_meta(seed, "checkpoint"),
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_loss": best_loss,
            "patience_counter": patience_counter,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "rng_state": rng,
            "seed": seed,
            "protocol_hash": PROTOCOL_HASH,
            "stoi": stoi,
            "itos": itos,
            "training_complete": False,
        }

        save_checkpoint(state, last_path)
        if is_best:
            save_checkpoint(state, best_path)
            print(f"Seed {seed} Epoch {epoch:02d}: Train_Loss={train_loss:.6f} • Val_Loss={val_loss:.6f} (Lưu Best!)")
        else:
            print(f"Seed {seed} Epoch {epoch:02d}: Train_Loss={train_loss:.6f} • Val_Loss={val_loss:.6f} (Patience={patience_counter}/{TRAIN_CONFIG['patience']})")

        if patience_counter >= TRAIN_CONFIG["patience"]:
            print(f"Seed {seed}: Dừng sớm tại Epoch {epoch} do không cải thiện liên tiếp {TRAIN_CONFIG['patience']} lần.")
            break

    check_lock(seed)
    # Đánh dấu hoàn tất huấn luyện thành công
    best_state = load_checkpoint(best_path, seed)
    best_state["training_complete"] = True
    save_checkpoint(best_state, best_path)

    last_state = load_checkpoint(last_path, seed)
    last_state["training_complete"] = True
    save_checkpoint(last_state, last_path)

    print(f"Seed {seed}: Huấn luyện HOÀN TẤT. Best Epoch: {best_state['best_epoch']} (Val Loss: {best_state['best_loss']:.6f})")
    del model, criterion, optimizer, best_state, last_state
    gc.collect()
    torch.cuda.empty_cache()

if SELECTED_SEED in (44, 46):
    with writing_scope(SELECTED_SEED):
        train_seed(SELECTED_SEED)
        evaluate_seed(SELECTED_SEED, seed_dir(SELECTED_SEED) / f"checkpoint_best_seed{SELECTED_SEED}.pt")
else:
    print(f"Seed hiện tại là {SELECTED_SEED}; Cell 5 chỉ thực thi khi chọn seed 44 hoặc 46.")

Seed 44: Huấn luyện đã hoàn thành từ trước.
Seed 44: đã có 5.000 predictions; chỉ chạy phần chấm điểm.
PTB tokenize toàn bộ 25.010 reference captions...
Đang chấm: Bleu_1, Bleu_2, Bleu_3, Bleu_4
{'testlen': 47397, 'reflen': 47257, 'guess': [47397, 42397, 37397, 32397], 'correct': [35049, 18802, 9300, 4615]}
ratio: 1.002962524070487
Đang chấm: METEOR
Đang chấm: ROUGE_L
Đang chấm: CIDEr
Đang chấm: SPICE
Seed 44 hoàn tất: {'Bleu_1': 0.739477, 'Bleu_2': 0.57266, 'Bleu_3': 0.433657, 'Bleu_4': 0.328304, 'METEOR': 0.266739, 'ROUGE_L': 0.549051, 'CIDEr': 1.074608, 'SPICE': 0.195034}


In [10]:
#@title CELL 6 — Trạng thái từng seed; chỉ tổng hợp khi đủ 3 kết quả cuối
def summarize_completed_seeds():
    ALL_SEEDS = [42, 44, 46]
    seed_metrics = {}
    source_hashes = {}
    for seed in ALL_SEEDS:
        path = seed_dir(seed) / f"metrics_seed{seed}.json"
        prediction_path = seed_dir(seed) / f"predictions_seed{seed}.json"
        if not path.is_file() or not prediction_path.is_file():
            raise FileNotFoundError(f"Thiếu kết quả seed {seed}; hãy chạy Cell 4/5 trước.")
        row = read_owned_json(path, seed, "metrics")
        meta = row.get("_meta", {})
        if not valid_metrics(row) or meta.get("seed") != seed:
            raise ValueError(f"Điểm seed {seed} thiếu hoặc không hợp lệ.")
        if meta.get("protocol_hash") != PROTOCOL_HASH:
            raise ValueError(f"Seed {seed} dùng protocol khác, không được gộp.")
        expected_checkpoint = (SEED42_CHECKPOINT if seed == 42 else
                               seed_dir(seed) / f"checkpoint_best_seed{seed}.pt")
        if Path(meta.get("checkpoint", "")).resolve() != expected_checkpoint.resolve():
            raise ValueError(f"Seed {seed} trỏ nhầm checkpoint.")
        signature = canonical_hash({"seed": seed,
            "checkpoint_sha256": meta["checkpoint_sha256"], "protocol_hash": PROTOCOL_HASH})
        if meta.get("signature") != signature or meta.get("status") != "complete":
            raise ValueError(f"Seed {seed} chưa có metrics cuối đúng signature.")
        packet = load_prediction_packet(prediction_path, seed, signature)
        if packet is None or not packet["_meta"]["complete"]:
            raise ValueError(f"Seed {seed} chưa có đủ 5.000 predictions cuối.")
        if meta.get("predictions_sha256") != sha256_file(prediction_path):
            raise ValueError(f"Predictions seed {seed} đã thay đổi sau khi chấm.")
        checkpoint_path = Path(meta["checkpoint"])
        if sha256_file(checkpoint_path) != meta["checkpoint_sha256"]:
            raise ValueError(f"Checkpoint seed {seed} đã thay đổi sau khi đánh giá.")
        source_hashes[path] = _OBSERVED_FILES[str(output_path(path))]
        source_hashes[prediction_path] = meta["predictions_sha256"]
        source_hashes[checkpoint_path] = meta["checkpoint_sha256"]
        seed_metrics[seed] = row

    statistics = {}
    for name in METRIC_NAMES:
        values = np.asarray([seed_metrics[s][name] for s in ALL_SEEDS], dtype=np.float64)
        statistics[name] = {"mean": float(values.mean()),
                            "std": float(values.std(ddof=1)),
                            "values": {str(s): float(v) for s, v in zip(ALL_SEEDS, values)}}
    summary = {
        "_meta": owned_meta(None, "summary"),
        "metrics_sha256": {str(s): source_hashes[
            seed_dir(s) / f"metrics_seed{s}.json"] for s in ALL_SEEDS},
        "seeds": ALL_SEEDS, "n_seeds": 3, "std_ddof": 1,
        "std_definition": "sample standard deviation across three seeds, not confidence interval",
        "metric_scale": "raw pycocoevalcap scores; multiply ALL values by 100 only if desired",
        "test_images": 5000, "decode": DECODE_CONFIG,
        "protocol_hash": PROTOCOL_HASH, "protocol": PROTOCOL,
        "training_note": "Historical Level2 seed42 checkpoint reused; seeds44/46 fine-tune 30 epochs max at lr3e-4 from fixed historical Level1 seed42. Conditional stage variation; historical loss/min_delta training source not fully proven; not end-to-end paired seed pipelines.",
        "comparison_note": "All seeds rescored with Java METEOR; old NLTK METEOR is not interchangeable.",
        "metrics": statistics,
    }
    for path, expected_hash in source_hashes.items():
        if sha256_file(path) != expected_hash:
            raise RuntimeError(f"Input tổng hợp vừa đổi; giữ nguyên summary: {path.name}")
    summary_path = RUN_DIR / "level2_3seeds_beam3_summary.json"
    existing = read_owned_json(summary_path, None, "summary")
    if existing is not None:
        if existing != summary:
            raise ValueError("Summary có sẵn khác nội dung; giữ nguyên, không ghi đè.")
        print("Summary đã có và khớp; không sửa file.")
    else:
        save_json(summary, summary_path)

    labels = {"Bleu_1": "BLEU-1", "Bleu_2": "BLEU-2", "Bleu_3": "BLEU-3",
              "Bleu_4": "BLEU-4", "METEOR": "METEOR", "ROUGE_L": "ROUGE-L",
              "CIDEr": "CIDEr", "SPICE": "SPICE"}
    markdown = [
        "| Chỉ số (thang gốc) | Seed 42 | Seed 44 | Seed 46 | Mean ± Std |",
        "|:--|--:|--:|--:|--:|",
    ]
    for name in METRIC_NAMES:
        st = statistics[name]
        cells = [labels[name]] + [f"{seed_metrics[s][name]:.4f}" for s in ALL_SEEDS]
        cells.append(f"{st['mean']:.4f} ± {st['std']:.4f}")
        markdown.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(markdown)))
    display(Markdown(
        "**Quy ước:** điểm gốc của pycocoevalcap, Std mẫu với ddof=1, n=3. "
        "Đây là biến thiên giữa các seed, không phải khoảng tin cậy. "
        "Level2 seed42 lịch sử được dùng lại; seed44/46 fine-tune tối đa 30 epoch, lr3e-4 từ cùng Level1 seed42 cố định. Loss/min_delta lịch sử chưa xác minh đầy đủ. Biến thiên có điều kiện theo stage, không phải ba pipeline end-to-end. "
        "METEOR cả ba seed dùng cùng Java scorer, khác cách tính NLTK cũ."
    ))

    latex = [
        r"\begin{table}[htbp]", r"\centering",
        r"\caption{Level 2 on COCO 2014 Karpathy Test (5,000 images), "
        r"beam size 3, $\alpha=0.6$. Raw scores; mean $\pm$ sample standard deviation ($n=3$).}",
        r"\label{tab:level2-multiseed-beam3}",
        r"\begin{tabular}{lrrrr}", r"\hline",
        r"Metric & Seed 42 & Seed 44 & Seed 46 & Mean $\pm$ Std \\",
        r"\hline",
    ]
    for name in METRIC_NAMES:
        st = statistics[name]
        cells = [labels[name]] + [f"{seed_metrics[s][name]:.4f}" for s in ALL_SEEDS]
        cells.append("$" + f"{st['mean']:.4f} \\pm {st['std']:.4f}" + "$")
        latex.append(" & ".join(cells) + r" \\")
    latex += [
        r"\hline", r"\end{tabular}", r"\par\smallskip",
        r"{\footnotesize Historical Level2 seed42 reused; fixed historical Level1 seed42 initialization for all new runs. "
        r"Seeds44/46: at most 30 fine-tuning epochs, lr3e-4; historical loss/min-delta recipe gaps disclosed. Conditional stage seeds, not end-to-end paired runs. All use Java METEOR.}",
        r"\end{table}",
    ]
    print("\nLaTeX để dán vào báo cáo:\n")
    print("\n".join(latex))
    print("\nĐã lưu:", summary_path)

missing_seeds = []
for seed in (42, 44, 46):
    folder = seed_dir(seed)
    metric_file = folder / f"metrics_seed{seed}.json"
    pred_file = folder / f"predictions_seed{seed}.json"
    if metric_file.is_file() and pred_file.is_file():
        print(f"Seed {seed}: có kết quả — sẽ xác minh hash/trạng thái trước khi gộp.")
    else:
        missing_seeds.append(seed)
        print(f"Seed {seed}: chưa có đầy đủ kết quả cuối. Thư mục:", folder)

if missing_seeds:
    print("\nChưa tổng hợp; đây là trạng thái bình thường khi chạy riêng từng seed.")
    print("Đổi SEED_TO_RUN ở Cell 2 để chạy các seed còn thiếu:", missing_seeds)
    print("Dữ liệu/điểm cũ không được sửa. Chạy lại Cell 6 sau khi đủ 42, 44, 46.")
else:
    with writing_scope("summary"):
        summarize_completed_seeds()


Seed 42: có kết quả — sẽ xác minh hash/trạng thái trước khi gộp.
Seed 44: có kết quả — sẽ xác minh hash/trạng thái trước khi gộp.
Seed 46: chưa có đầy đủ kết quả cuối. Thư mục: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level2/multiseed_beam3_v1/run_3aace7f5829acedd/seed_46

Chưa tổng hợp; đây là trạng thái bình thường khi chạy riêng từng seed.
Đổi SEED_TO_RUN ở Cell 2 để chạy các seed còn thiếu: [46]
Dữ liệu/điểm cũ không được sửa. Chạy lại Cell 6 sau khi đủ 42, 44, 46.
